# Triangle random intersection graphs: compare graph sizes

This notebook explores the measurements saved by
`03_triangle_simulation.ipynb`. We compare increasing graph sizes, look more
closely at the rise in the 3-core fraction, and inspect individual outcomes
that an average might hide. All measurements are read from CSV files: running
this notebook never constructs a graph or starts a simulation.

Select the project's `.venv` kernel, review **Analysis settings**, and run the
cells in order. Start with the `pilot` results, then analyse `comparison` and
`refinement` after collecting those stages in notebook 03. Each stage has its
own output filenames, so analysing one stage does not replace another's figures.

In [ ]:
%matplotlib inline

import csv
from pathlib import Path

import numpy as np
from IPython.display import display
from matplotlib.figure import Figure

# Find the project root whether the kernel starts here or one folder above.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
PROJECT_ROOT = NOTEBOOK_DIR.parent

## Analysis settings

Choose the same `data_dir` and `stage` as notebook 03. We read every
`<stage>_n<size>.csv` in that directory; each file contains one graph size.
Keep unrelated runs in different directories so they are not combined by
accident. All paths below start from the project root.

The smoothing window controls the marker for the steepest rise, not the
plotted measurements. It must be a positive odd integer. We also repeat the
calculation with `sensitivity_windows` and with every second lambda point,
to see how much this marker depends on our analysis choices.

`zoom_width` is the distance in lambda to show on either side of the rise
estimated for the largest graph size. For the histograms, `None` selects that
largest size and three nearby sampled lambda values. You can instead choose
a particular size and a list of lambda values already present in the data.

In [ ]:
config = {
    "data_dir": "results/runs/triangle_study",  # Directory written by notebook 03.
    "stage": "pilot",  # Analyse pilot, comparison, or refinement measurements.
    "figure_dir": "results/figures/triangle_study",  # Figures and summary tables.
    "smoothing_window": 3,  # Odd number of points used to locate the steepest rise.
    "sensitivity_windows": [1, 3, 7],  # Compare unsmoothed and smoothed estimates.
    "zoom_width": 0.3,  # Lambda distance on each side of the estimated rise.
    "histogram_n": None,  # None chooses the largest graph size in this stage.
    "histogram_lambdas": None,  # None chooses three sampled values near its rise.
}
config

## Read the saved trials

Each row represents one sampled graph. Its seed and simulation parameters
allow that trial to be repeated in notebook 03. We check that the core
fraction agrees with the saved number of surviving vertices. Seeds are
read as integers so their exact values are preserved.

The same graph size, lambda value, and replicate number must not appear
twice in the selected files. Counting a duplicated trial twice would make
the sample look larger than it really is, so we raise an error instead.
If measurements are missing, collect them in notebook 03 before returning here.

In [ ]:
def load_results(path: str | Path) -> list[dict]:
    """Read one CSV and check that its measurements are consistent."""
    fields = ["model", "n", "lambda", "replicate", "seed", "core_size", "core_fraction"]
    rows = []
    with Path(path).open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != fields:
            raise ValueError(f"Expected CSV columns: {', '.join(fields)}")
        for row in reader:
            for key in ("n", "replicate", "seed", "core_size"):
                row[key] = int(row[key])
            row["lambda"] = float(row["lambda"])
            row["core_fraction"] = float(row["core_fraction"])
            if (
                row["model"] != "triangle_rig"
                or row["n"] <= 0
                or not 0 <= row["core_size"] <= row["n"]
                or not np.isfinite(row["lambda"])
                or row["lambda"] < 0
                or row["replicate"] < 0
                or row["seed"] < 0
                or not np.isclose(
                    row["core_fraction"],
                    row["core_size"] / row["n"],
                    rtol=1e-9,
                    atol=1e-12,
                )
            ):
                raise ValueError("Invalid measurement or inconsistent core fraction")
            rows.append(row)
    if not rows:
        raise ValueError(f"The CSV contains no measurements: {path}")
    return rows


def load_study(directory: str | Path, stage: str) -> list[dict]:
    """Combine a stage's saved files without counting any trial twice."""
    paths = sorted(Path(directory).glob(f"{stage}_n*.csv"))
    if not paths:
        raise FileNotFoundError(
            f"No {stage!r} measurements in {directory}. "
            "Select this stage in notebook 03 and run it first."
        )
    rows = []
    seen = set()
    for path in paths:
        for row in load_results(path):
            trial = (row["n"], row["lambda"], row["replicate"])
            if trial in seen:
                raise ValueError(f"Duplicate trial {trial} in {path}")
            seen.add(trial)
            rows.append(row)
    return rows

## Summarize variation between graphs

At each graph size and lambda value we calculate the mean, median, and
the 25th and 75th percentiles of the core fractions. The mean is their
average; the median is their middle value. The interval between those
percentiles covers the middle half of the sampled outcomes.

That interval describes **variation between graphs**. It is not a
confidence interval for the mean. We separately record the estimated
standard error of the mean,
$s/\sqrt{r}$, where $r$ is the number of independent trials and $s$ is
their sample standard deviation. This describes sampling uncertainty
in the average; it is undefined when there is only one trial. It does
not measure bias from using finite graphs or a coarse lambda grid.

Near the transition, a smooth average may hide a mixture of nearly empty
cores and substantial cores. Later histograms show those outcomes directly.

In [ ]:
def summarize_results(rows: list[dict]) -> list[dict]:
    """Summarize each graph size and lambda, regardless of input order."""
    if not rows:
        raise ValueError("Provide at least one measurement")
    grouped = {}
    for row in rows:
        grouped.setdefault((row["n"], row["lambda"]), []).append(row["core_fraction"])

    summary = []
    for (n, lam), sample in sorted(grouped.items()):
        summary.append(
            {
                "n": n,
                "lambda": lam,
                "trials": len(sample),
                "mean": float(np.mean(sample)),
                "median": float(np.median(sample)),
                "q25": float(np.percentile(sample, 25)),
                "q75": float(np.percentile(sample, 75)),
                "minimum": float(min(sample)),
                "maximum": float(max(sample)),
                "standard_error": (
                    float(np.std(sample, ddof=1) / np.sqrt(len(sample)))
                    if len(sample) > 1
                    else None
                ),
            }
        )
    return summary


def estimate_transition(lambdas, means, window: int = 3) -> float | None:
    """Find the steepest positive interior slope of the smoothed means.

    This describes the sampled finite graphs. Return None for a short,
    flat, or decreasing curve rather than inventing a transition location.
    """
    if not isinstance(window, (int, np.integer)) or window < 1 or window % 2 == 0:
        raise ValueError("The smoothing window must be a positive odd integer")
    lambdas = np.asarray(lambdas, dtype=float)
    means = np.asarray(means, dtype=float)
    if (
        lambdas.ndim != 1
        or means.ndim != 1
        or len(lambdas) != len(means)
        or not np.all(np.isfinite(lambdas))
        or not np.all(np.isfinite(means))
        or np.any(np.diff(lambdas) <= 0)
    ):
        raise ValueError("Provide finite means on a strictly increasing lambda grid")
    if len(lambdas) < 3 or np.ptp(means) <= np.finfo(float).eps:
        return None

    # Keep the window odd and small enough to fit the available grid.
    # Repeat endpoint values so there is one smoothed mean per lambda.
    window = min(window, len(lambdas) if len(lambdas) % 2 else len(lambdas) - 1)
    padded = np.pad(means, window // 2, mode="edge")
    smoothed = np.convolve(padded, np.ones(window) / window, mode="valid")
    slopes = np.gradient(smoothed, lambdas)
    index = int(np.argmax(slopes[1:-1])) + 1
    return float(lambdas[index]) if slopes[index] > 0 else None


def transition_summary(rows: list[dict], windows=(1, 3, 7)) -> list[dict]:
    """Check slope estimates across sizes, smoothing, and grid spacing."""
    summary = summarize_results(rows)
    estimates = []
    for n in sorted({row["n"] for row in summary}):
        measurements = [row for row in summary if row["n"] == n]
        for window in windows:
            for stride in (1, 2):
                selected = measurements[::stride]
                lambdas = [row["lambda"] for row in selected]
                means = [row["mean"] for row in selected]
                estimate = estimate_transition(lambdas, means, window)
                gaps = np.diff(lambdas)
                largest_odd_window = len(lambdas) - (len(lambdas) % 2 == 0)
                estimates.append(
                    {
                        "n": n,
                        "smoothing_window": window,
                        "effective_window": min(window, largest_odd_window),
                        "grid_stride": stride,
                        "lambda_points": len(lambdas),
                        "lambda_step_min": float(min(gaps)) if len(gaps) else None,
                        "lambda_step_max": float(max(gaps)) if len(gaps) else None,
                        "slope_estimate": estimate,
                    }
                )
    return estimates

## Compare graph sizes and look more closely at the rise

For each size, solid lines show the mean, dashed lines show the median,
and the shaded band covers the middle half of the observed fractions.
The vertical dotted line marks where that size's smoothed mean has its
largest positive interior slope. Smoothing affects only this marker.

The first plot shows the whole sampled range. The second uses the same
measurements but narrows the horizontal axis around the largest size's
estimated rise. When no rise can be estimated, it centres on the middle
of that size's sampled lambda range instead.

If the rise lies near a boundary, collect a wider lambda range before
interpreting it. A zoom cannot reveal behaviour outside the saved data.

In [ ]:
def plot_size_comparison(
    rows: list[dict], smoothing_window: int = 3, zoom_width: float | None = None
) -> Figure:
    """Compare means, medians, and variation for every saved graph size."""
    summary = summarize_results(rows)
    sizes = sorted({row["n"] for row in summary})
    figure = Figure(figsize=(10, 6), layout="constrained")
    axis = figure.subplots()
    for index, n in enumerate(sizes):
        measurements = [row for row in summary if row["n"] == n]
        lambdas = [row["lambda"] for row in measurements]
        means = [row["mean"] for row in measurements]
        color = f"C{index % 10}"
        counts = [row["trials"] for row in measurements]
        trials = (
            str(counts[0])
            if min(counts) == max(counts)
            else f"{min(counts)}–{max(counts)}"
        )
        axis.fill_between(
            lambdas,
            [row["q25"] for row in measurements],
            [row["q75"] for row in measurements],
            color=color,
            alpha=0.18,
        )
        axis.plot(
            lambdas,
            means,
            color=color,
            linewidth=2,
            label=f"n={n:,}; {trials} trials/λ",
        )
        axis.plot(
            lambdas,
            [row["median"] for row in measurements],
            color=color,
            linestyle="--",
            linewidth=1.2,
        )
        estimate = estimate_transition(lambdas, means, smoothing_window)
        if estimate is not None:
            axis.axvline(estimate, color=color, linestyle=":", alpha=0.75)

    if zoom_width is not None:
        if not np.isfinite(zoom_width) or zoom_width <= 0:
            raise ValueError("The zoom width must be positive")
        largest = [row for row in summary if row["n"] == sizes[-1]]
        lambdas = [row["lambda"] for row in largest]
        center = estimate_transition(
            lambdas, [row["mean"] for row in largest], smoothing_window
        )
        if center is None:
            center = (lambdas[0] + lambdas[-1]) / 2
        lower = max(min(row["lambda"] for row in summary), center - zoom_width)
        upper = min(max(row["lambda"] for row in summary), center + zoom_width)
        if lower < upper:
            axis.set_xlim(lower, upper)
    else:
        lambdas = [row["lambda"] for row in summary]
        if min(lambdas) < max(lambdas):
            axis.set_xlim(min(lambdas), max(lambdas))

    view = (
        "Closer view of the rise"
        if zoom_width is not None
        else "Comparison across graph sizes"
    )
    axis.set_title(
        f"{view}\nSolid: mean; dashed: median; shading: middle half of graphs"
    )
    axis.set_xlabel(r"$\lambda$")
    axis.set_ylabel(r"3-core fraction $|\mathrm{Core}_3(G)|/n$")
    axis.set_ylim(0, 1)
    axis.grid(True, alpha=0.25)
    axis.legend()
    return figure


def plot_distributions(
    rows: list[dict], n: int | None = None, lambdas=None, smoothing_window: int = 3
) -> Figure:
    """Show individual core-fraction distributions at selected lambda values."""
    summary = summarize_results(rows)
    sizes = sorted({row["n"] for row in summary})
    n = sizes[-1] if n is None else n
    if n not in sizes:
        raise ValueError(f"No measurements for n={n}")
    measurements = [row for row in summary if row["n"] == n]
    available = np.array([row["lambda"] for row in measurements])
    if lambdas is None:
        center = estimate_transition(
            available, [row["mean"] for row in measurements], smoothing_window
        )
        if center is None:
            center = (available[0] + available[-1]) / 2
        # Choose up to three actual sampled points, ordered from left to right.
        selected = sorted(available[np.argsort(abs(available - center))[:3]])
    else:
        selected = []
        for lam in lambdas:
            closest = float(available[np.argmin(abs(available - lam))])
            if not np.isclose(closest, lam, rtol=1e-9, atol=1e-10):
                raise ValueError(f"Lambda {lam} was not sampled for n={n}")
            selected.append(closest)
        if not selected or len(set(selected)) != len(selected):
            raise ValueError("Choose at least one lambda value, without duplicates")

    figure = Figure(figsize=(4 * len(selected), 4), layout="constrained")
    axes = figure.subplots(1, len(selected), sharex=True, sharey=True, squeeze=False)[0]
    # Use the same bins and axis limits so panels can be compared directly.
    bins = np.linspace(0, 1, 21)
    for axis, lam in zip(axes, selected):
        sample = [
            row["core_fraction"]
            for row in rows
            if row["n"] == n and row["lambda"] == lam
        ]
        axis.hist(sample, bins=bins, color="tab:blue", edgecolor="white")
        axis.axvline(np.mean(sample), color="tab:orange", linestyle="--", label="Mean")
        axis.set_title(f"λ={lam:.4g}\nn={n:,}; {len(sample)} trials")
        axis.set_xlabel("3-core fraction")
        axis.set_xlim(0, 1)
        axis.grid(True, axis="y", alpha=0.25)
    axes[0].set_ylabel("Number of sampled graphs")
    axes[0].legend()
    figure.suptitle("Individual outcomes near the rise")
    return figure


def save_table(rows: list[dict], path: str | Path) -> None:
    """Save a summary table; unavailable estimates become empty CSV cells."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
rows = load_study(PROJECT_ROOT / config["data_dir"], config["stage"])
summary = summarize_results(rows)
sizes = sorted({row["n"] for row in rows})
print(
    f"Loaded {len(rows)} trials for graph sizes {sizes} in stage {config['stage']!r}."
)

## Draw and save the figures

The core fraction is the number of surviving vertices divided by the
total number $n$ of individuals. Zero means an empty core; one means that
every vertex survives. A nonempty core containing only a few vertices
is different from one containing a substantial fraction of the graph.

Compare how the rise changes as $n$ grows, then inspect the histograms.
Do they suggest fractions growing gradually from zero, or a mixture of
nearly empty and substantial cores? Neither a smooth average nor a single
large graph settles the limiting behaviour by itself.

The next cell saves three PNG files, replacing earlier figures for this
stage. It leaves all raw CSV measurements untouched.

In [ ]:
figure_dir = PROJECT_ROOT / config["figure_dir"]
figure_dir.mkdir(parents=True, exist_ok=True)
figures = {
    "comparison": plot_size_comparison(rows, config["smoothing_window"]),
    "zoom": plot_size_comparison(
        rows, config["smoothing_window"], config["zoom_width"]
    ),
    "distributions": plot_distributions(
        rows,
        config["histogram_n"],
        config["histogram_lambdas"],
        config["smoothing_window"],
    ),
}
for name, figure in figures.items():
    path = figure_dir / f"{config['stage']}_{name}.png"
    figure.savefig(path, dpi=160)
    display(figure)
    print(f"Saved {path}.")

## Check sensitivity to smoothing and lambda spacing

The maximum-slope location is a **finite-size description**, not a proof
of a critical parameter as $n\to\infty$. Its location can change with
graph size, the number of trials, lambda spacing, and smoothing.

For each size we calculate it using several smoothing windows, first
with every saved lambda value (`grid_stride=1`) and then with every
second value (`grid_stride=2`). The second calculation deliberately
uses a coarser subset of the same data. This is a sensitivity check,
not an independent experiment or an uncertainty interval.

A window is shortened to fit a small grid. The CSV records both the
requested and effective window, and the smallest and largest actual
lambda gaps. Blank estimates mean that the curve is too short, flat,
or has no positive interior slope. Large changes across these choices
suggest collecting a finer grid or more trials before drawing conclusions.

We also save all means, medians, percentiles, and standard errors in
`<stage>_summary.csv`. The estimate table below is saved separately in
`<stage>_transition_estimates.csv`.

In [ ]:
estimates = transition_summary(rows, config["sensitivity_windows"])
save_table(summary, figure_dir / f"{config['stage']}_summary.csv")
save_table(estimates, figure_dir / f"{config['stage']}_transition_estimates.csv")

print("      n  window  stride  lambda points  slope location")
for estimate in estimates:
    location = estimate["slope_estimate"]
    text = "unavailable" if location is None else f"{location:.5f}"
    print(
        f"{estimate['n']:7d}  {estimate['effective_window']:6d}  "
        f"{estimate['grid_stride']:6d}  {estimate['lambda_points']:13d}  {text}"
    )
print(f"Saved summary tables to {figure_dir}.")

## Record what the study shows

After collecting the comparison and refinement stages, use their plots
and tables to write a short interpretation:

1. Where is the rise for each size, and does its location appear to settle?
2. Does the change become sharper as the graphs grow? Do the individual
   outcomes suggest gradual emergence or a jump to a positive fraction?
3. Does that interpretation survive changes in smoothing and lambda spacing?
   Do more trials change the mean or reveal a mixture of outcomes?
4. Which conclusions are supported by these measurements, and what remains
   uncertain because of finite sizes, sampling, or the chosen lambda range?

Keep observations separate from conjectures about the limiting model.
Later configuration-model comparisons and cavity equations will provide
different evidence about the role of triangles and the predicted transition.